In [ ]:
#| default_exp cli

# A command line for reading

Everything so far assumes you're already inside SolveIt: `add_msg` appends blocks to the dialog you're reading in. But choosing *what* to read is a browsing task, and browsing is pleasant in a terminal — especially when the answer is sitting in a Zotero inbox of fifty papers.

So this module offers the same pipeline as three commands:

```
zot-ls                     # which collections do I have?
zot-ls '0.1 Inbox'         # what's in this one?
zot-nb 3                   # turn paper 3 into a notebook
paper-nb https://arxiv.org/abs/2504.20997   # or convert a public URL
```

In [ ]:
#| export
# A terminal front end: browse a Zotero library and turn its papers into SolveIt notebooks.
import json, re, asyncio, os, httpx
from pathlib import Path
from xml.etree import ElementTree
from fastcore.script import call_parse

from kmacsolves.paper2solveit import (list_zotero, zotero_download, solve_markdown_paper,
    pdf2md, download_to_temp, _zotero_client, _describe, _authors, _year, _label)

The trick is that `solve_markdown_paper` doesn't care where its blocks end up — it just awaits `add`. Inside SolveIt that's `add_msg`, writing into the live dialog. Here it's a `NotebookBuilder`, collecting the same blocks into a file. The splitting, the citation footnotes, the collapsed bibliography: all of it is the code that already runs in SolveIt, so a notebook built from the terminal is the same artifact you'd have made by hand.

`add_msg`'s two placements are the whole interface. `at_end` appends; `add_after` slots a section's citation cheatsheet in just below its header. `i_collapsed` becomes `hide_input`, which is how SolveIt renders a folded cell.

In [ ]:
#| export
# SolveIt's own notebook-level settings, copied from a dialog it wrote itself.
SOLVEIT_META = {'solveit': {'default_code': True, 'mode': 'learning',
                            'use_thinking': True, 'use_tools': True, 'ver': 2}}

class NotebookBuilder:
    """Collects `solve_markdown_paper` blocks into a notebook file instead of a live dialog.

    Its `add` method stands in for `dialoghelper.add_msg`, so the same splitting and citation
    logic that runs inside SolveIt produces the file you get from the terminal.
    """
    def __init__(self, header=()):
        self.cells = [self._cell(h, 'code') for h in header]
    def _cell(self, content, kind='markdown', collapsed=False):
        self._n = getattr(self, '_n', 0) + 1
        cell = {'cell_type': kind, 'id': f'p2s{self._n:04d}',
                'metadata': {'hide_input': True} if collapsed else {},
                'source': content.splitlines(keepends=True)}
        if kind == 'code': cell |= {'execution_count': None, 'outputs': []}
        return cell
    async def add(self, content, placement='at_end', id=None, i_collapsed=None):
        "Stand-in for `add_msg`: append a block, or slot one in after an earlier block."
        cell = self._cell(content, collapsed=bool(i_collapsed))
        if placement == 'add_after':
            at = next(i for i, c in enumerate(self.cells) if c['id'] == id)
            self.cells.insert(at + 1, cell)
        else: self.cells.append(cell)
        return cell['id']
    def write(self, path):
        "Save as a notebook SolveIt will open in learning mode."
        nb = {'cells': self.cells, 'metadata': SOLVEIT_META, 'nbformat': 4, 'nbformat_minor': 5}
        Path(path).write_text(json.dumps(nb, indent=1, ensure_ascii=False) + '\n')
        return Path(path)

Notebooks name themselves after the paper, in the style of an existing reading pile — `Vaswani17 - Attention Is All You Need.ipynb` — with anything that would upset a filesystem stripped out.

In [ ]:
#| export
def nb_filename(data, path='.'):
    "A filename in the reading-nbs style: 'Vaswani17 - Attention Is All You Need.ipynb'."
    who = re.sub(r'\s*(et al\.|&.*)$', '', _authors(data)).strip()
    title = data.get('title') or data.get('filename') or 'Paper'
    stem = f"{who}{_year(data)[2:]} - {title}" if who else title
    stem = re.sub(r'[\\/:*?"<>|]', '-', stem).strip()[:120]
    return Path(path) / f"{stem}.ipynb"

One wrinkle the notebook API doesn't have: each terminal command is its own process, so the in-memory listing that `zotero_pdf(3)` relies on is gone by the time you type the next command. `zot-ls` therefore writes what it showed you to a cache file, including which attachment it resolved for each entry, and `zot-nb` reads its numbers back from there.

The obvious caveat applies: the numbers refer to the last listing you ran, not to the library as it stands now. Re-run `zot-ls` if you've been editing Zotero in between.

In [ ]:
#| export
CACHE = Path(os.environ.get('XDG_CACHE_HOME', Path.home() / '.cache')) / 'kmacsolves' / 'listing.json'

def _save_listing(listing, local):
    "Remember what `zot_ls` just showed, so `zot_nb` in a later process can resolve its numbers."
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    CACHE.write_text(json.dumps({
        'kind': listing.kind, 'title': listing.title, 'local': local,
        'entries': [{'key': e['key'], 'label': _label(e), 'data': e['data'],
                     'pdf': listing.pdfs.get(e['key'])} for e in listing.entries]}))

def _load_listing():
    "The listing `zot_ls` last wrote, or a clear error if there isn't one."
    if not CACHE.exists(): raise SystemExit(
        "Nothing listed yet — run `zot-ls <collection>` first, or pass a Zotero item key.")
    return json.loads(CACHE.read_text())

def _cached_entry(n):
    "Entry `n` from the saved listing, checking it's a paper rather than a collection."
    d = _load_listing()
    if d['kind'] != 'item': raise SystemExit(
        f"The last listing was of collections. Run `zot-ls <collection>` to list papers first.")
    if not 1 <= n <= len(d['entries']): raise SystemExit(
        f"{n} is outside the last listing of {len(d['entries'])} items ({d['title']}).")
    return d['entries'][n-1], d

`zot-ls` is the browsing half. It defaults to the desktop app, since that's the common case at a terminal and needs no credentials; `--web` switches to the Zotero web API for a machine without Zotero installed.

In [ ]:
#| export
@call_parse(pos=['collection'])
def zot_ls(
    collection: str = None,   # Collection name, or its number from a previous listing
    web: bool = False,        # Read via the Zotero web API instead of the desktop app
):
    "List Zotero collections, or the papers in one, numbered for `zot-nb`."
    kw = {'local': not web}
    listing = list_zotero(int(collection) if (collection or '').isdigit() else collection, **kw)
    _save_listing(listing, kw['local'])
    print(repr(listing))
    if listing.kind == 'collection': print("\nList one with `zot-ls <name-or-number>`.")
    else: print(f"\nConvert one with `zot-nb <number>`.  ({len(listing.pdfs)} of "
                f"{len(listing)} have a PDF)")

And the converting half. Taking the attachment key from the cached listing means we already know exactly which PDF to fetch, so this is one request rather than a re-listing. Marker also extracts the paper's figures, and the markdown refers to them by a relative `figures/` path, so they're written to the same directory as the notebook — otherwise every image breaks the moment you convert into somewhere other than the directory you're standing in.

In [ ]:
#| export
def _md_title(md):
    "A paper's own title, taken from the first heading of the converted markdown."
    m = re.search(r'^#+\s*(.+)', md, re.M)
    return m.group(1).strip() if m else None

async def _build(pdf, data, out, path, **kwargs):
    "Shared tail of the two converters: OCR to markdown, then lay it out as a notebook."
    # Extracted figures are referenced as 'figures/x.jpg', relative to the notebook — so they
    # have to be written beside it rather than into whatever directory you happened to run from.
    dest = Path(out).parent if out else Path(path)
    dest.mkdir(parents=True, exist_ok=True)
    o = await pdf2md(str(pdf), path=dest)
    # A URL carries no metadata, so fall back to whatever the paper calls itself.
    if not data.get('title'): data = data | {'title': _md_title(o["markdown"]) or 'Paper'}
    b = NotebookBuilder(header=['from kmacsolves.paper2solveit import *',
                                f'# {_describe(data)}'])
    await solve_markdown_paper(o["markdown"], add=b.add, **kwargs)
    return b.write(out or nb_filename(data, path))

@call_parse
def zot_nb(
    item: str,                # Number from `zot-ls`, or a Zotero item key
    out: str = None,          # Write here instead of an auto-named file
    path: str = '.',          # Directory for the auto-named file
    web: bool = False,        # Read via the Zotero web API instead of the desktop app
):
    "Turn a paper in your Zotero library into a SolveIt notebook."
    if item.isdigit():
        entry, listing = _cached_entry(int(item))
        if not entry['pdf']: raise SystemExit(
            f"{entry['label']} has no PDF attached — `zot-ls` marks these '(no PDF)'.")
        # The listing already worked out which attachment to fetch, so go straight to it.
        key, data, local = entry['pdf'], entry['data'], listing['local']
    else:
        local = not web
        data = _zotero_client(local=local).item(item)['data']
        key = item
    print(f"Converting {_describe(data)} …")
    pdf = zotero_download(key, local=local)
    print(f"Wrote {asyncio.run(_build(pdf, data, out, path))}")

`paper-nb` is the same thing for a public URL — what `paper2solveit` does, but written to a file instead of into a dialog. A URL tells you nothing about authorship, so for arXiv links we ask arXiv, and end up with the same `Yuksekgonul26 - Learning to Discover at Test Time` naming a Zotero item would have produced. For any other link, or if the lookup fails, the notebook falls back to whatever heading the paper opens with.

In [ ]:
#| export
ATOM = '{http://www.w3.org/2005/Atom}'

def arxiv_id(url):
    "The bare arXiv id in a URL — '2601.16175' or 'math/0601001' — or None if it isn't one."
    m = re.search(r'arxiv\.org/(?:abs|pdf)/(.+?)(?:v\d+)?(?:\.pdf)?/?$', url, re.I)
    return m.group(1) if m else None

def arxiv_meta(url):
    """Title, authors and date for an arXiv URL, in the shape a Zotero item would give.

    A bare URL leaves a notebook nothing to name itself after, so ask arXiv. Returns an
    empty dict for anything that isn't an arXiv link, or if the lookup doesn't work out —
    the caller falls back to the paper's own first heading.
    """
    aid = arxiv_id(url)
    if not aid: return {}
    try:
        r = httpx.get('https://export.arxiv.org/api/query', params={'id_list': aid},
                      timeout=30, follow_redirects=True)
        e = ElementTree.fromstring(r.text).find(f'{ATOM}entry')
        if e is None: return {}
        def txt(tag, node=None):
            n = (node if node is not None else e).find(f'{ATOM}{tag}')
            return ' '.join(n.text.split()) if n is not None and n.text else ''
        names = [txt('name', a) for a in e.findall(f'{ATOM}author')]
        return {'title': txt('title'), 'date': txt('published'),
                'creators': [{'creatorType': 'author', 'lastName': n.split()[-1]}
                             for n in names if n.split()]}
    except Exception: return {}

@call_parse
def paper_nb(
    url: str,                 # arXiv abstract URL, or a direct link to a PDF
    out: str = None,          # Write here instead of an auto-named file
    path: str = '.',          # Directory for the auto-named file
):
    "Turn a paper at a public URL into a SolveIt notebook."
    f = download_to_temp(url)
    print(f"Wrote {asyncio.run(_build(f, arxiv_meta(url), out, path))}")

The Zotero and OCR legs need a library and an API key, but the part that's new here — laying blocks out as a notebook — needs neither, so it's worth checking directly.

In [ ]:
#| hide
# Check the notebook-building half without Zotero, a Datalab key, or a live dialog.
import asyncio

_paper = """# A Paper

Written by someone, following (Brown, 2019).

## 1 Introduction

Prior work (Smith, 2020) established the thing.

### 1.1 Background

Later work (Jones & Lee, 2021) refined it.

## References

Smith, J. (2020). A thing.
"""

_b = NotebookBuilder(header=['from kmacsolves.paper2solveit import *'])
asyncio.run(solve_markdown_paper(_paper, add=_b.add))
_texts = [''.join(c['source']) for c in _b.cells]
assert _b.cells[0]['cell_type'] == 'code', "the header cells survive"
assert _texts[1].startswith('### Full References')
assert _b.cells[1]['metadata'] == {'hide_input': True}, "bibliography is collapsed"
assert any('Prior work [2] established' in t for t in _texts), "citations become footnotes"

# A section's cheatsheet is tucked in just below its header, collapsed. The splitter demotes
# headers one level, so what was '## 1 Introduction' arrives here as '# 1 Introduction' —
# top-level sections must collect citations just as subsections do.
for _hdr, _cite in (('# 1 Introduction', '[2]: (Smith, 2020)'),
                    ('## 1.1 Background', '[3]: (Jones & Lee, 2021)')):
    _i = _texts.index(_hdr)
    assert _b.cells[_i+1]['metadata'] == {'hide_input': True}, f"cheatsheet follows {_hdr}"
    assert _cite in _texts[_i+1], f"{_hdr} collects its own citations"
# The title block is several lines, not a header, so its citations still get footnoted.
assert 'following [1].' in _texts[2], "preamble is content, not a section"
assert len({c['id'] for c in _b.cells}) == len(_b.cells), "cell ids are unique"

_out = _b.write('/tmp/_p2s_test.ipynb')
_nb = json.loads(_out.read_text())
assert _nb['metadata'] == SOLVEIT_META and _nb['nbformat'] == 4
_out.unlink()

assert str(nb_filename({'title': 'Attention Is All You Need', 'date': '2017-06-12',
    'creators': [{'creatorType': 'author', 'lastName': n} for n in ('Vaswani', 'Shazeer', 'Parmar')]}
    )) == 'Vaswani17 - Attention Is All You Need.ipynb'
assert str(nb_filename({'title': 'A/B: testing'})) == 'A-B- testing.ipynb', "path-safe, and no author prefix when there's no author"
print('cli helpers: ok')

# A paper that cites as [12] keeps its own numbering, and its cheatsheets quote the
# bibliography rather than a compressed (Author, year).
_numeric = """# N

## 1 Intro

Established by [1], extended in [2, 3] and surveyed in [5-6].

## References

- [1] Alice A. First thing. 2019.
- [2] Bob B. Second thing. 2020.
- [3] Carol C. Third thing. 2021.
- [5] Erin E. Fifth thing. 2023.
- [6] Frank F. Sixth thing. 2024.
"""
_b2 = NotebookBuilder(header=[])
asyncio.run(solve_markdown_paper(_numeric, add=_b2.add))
_t2 = [''.join(c['source']) for c in _b2.cells]
assert 'Established by [1], extended in [2, 3] and surveyed in [5-6].' in _t2, "text left alone"
_j = _t2.index('# 1 Intro')
_sheet = _t2[_j+1]
assert '[1]: Alice A. First thing. 2019.' in _sheet, "cheatsheet quotes the real entry"
for _n in ('[2]:', '[3]:', '[5]:', '[6]:'): assert _n in _sheet, f"{_n} expanded from a list or range"
assert '[4]:' not in _sheet, "uncited entries stay out"
assert not any(t.startswith('### Citations') for t in _t2[_t2.index('# References'):]), \
    "the bibliography doesn't get a cheatsheet repeating itself"

from kmacsolves.paper2solveit import extract_references, cited_numbers

# Heading level of the References section shouldn't matter — papers use ##, books #.
assert extract_references("# Refs\n\n## References\n\n- [1] X.\n\n# Next\n\nbody").strip() == '- [1] X.'
assert cited_numbers('see [3], [5-7] and [9, 11]') == {3, 5, 6, 7, 9, 11}
print('citation handling: ok')

# arXiv ids, offline — the lookup itself needs the network, so only the parsing is asserted.
assert arxiv_id('https://arxiv.org/abs/2601.16175') == '2601.16175'
assert arxiv_id('https://arxiv.org/abs/2601.16175v2') == '2601.16175', 'version suffix dropped'
assert arxiv_id('https://arxiv.org/pdf/2601.16175.pdf') == '2601.16175'
assert arxiv_id('https://arxiv.org/abs/math/0601001') == 'math/0601001', 'old-style id'
assert arxiv_id('https://example.com/paper.pdf') is None
assert arxiv_meta('https://example.com/paper.pdf') == {}, 'non-arXiv makes no request'
print('arxiv metadata: ok')

These are wired up as console scripts in `pyproject.toml`, so `pip install -e .` puts `zot-ls`, `zot-nb` and `paper-nb` on your PATH.

Reading from the Zotero desktop app needs its local API switched on: **Settings → Advanced → Allow other applications on this computer to communicate with Zotero**. Nothing else to configure — no API key, no user id. Conversion still needs `DATALAB_KEY` in your environment, as it does everywhere else.